# Mortgage Rate vs. Sales Volume

This notebook explores whether the national 30-year fixed mortgage rate appears related to sold transaction volume.

Important note: this is exploratory analysis, not causal proof. Mortgage rates are only one factor affecting transaction volume.

## 1. Import Packages And Set Paths

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "data" / "reports" / "mortgage_rate_vs_sales_volume"
FIGURE_DIR = REPORT_DIR / "figures"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SOLD_WITH_RATES_FILE = PROCESSED_DIR / "crmls_sold_combined_residential_with_mortgage_rates_202401_202605.csv"
SOLD_WITH_RATES_FILE

## 2. Load Enriched Sold Dataset

In [ ]:
sold = pd.read_csv(SOLD_WITH_RATES_FILE, low_memory=False)

print(f"Rows: {sold.shape[0]:,}")
print(f"Columns: {sold.shape[1]:,}")
sold[["CloseDate", "year_month", "ClosePrice", "rate_30yr_fixed"]].head()

## 3. Validate Mortgage Rate Merge

In [ ]:
print("Missing mortgage rates:", sold["rate_30yr_fixed"].isna().sum())
print("Missing year_month:", sold["year_month"].isna().sum())

sold[["year_month", "rate_30yr_fixed"]].drop_duplicates().sort_values("year_month").head()

## 4. Create Monthly Sales Volume

成交量 here means the number of Residential sold records per month.

In [ ]:
sold["CloseDateParsed"] = pd.to_datetime(sold["CloseDate"], errors="coerce")
sold["year_month"] = sold["CloseDateParsed"].dt.to_period("M").astype(str)

monthly_volume = (
    sold.dropna(subset=["CloseDateParsed", "rate_30yr_fixed"])
    .groupby("year_month")
    .agg(
        sales_volume=("ListingKey", "size"),
        rate_30yr_fixed=("rate_30yr_fixed", "mean"),
        median_close_price=("ClosePrice", "median"),
    )
    .reset_index()
)

monthly_volume["month_date"] = pd.to_datetime(monthly_volume["year_month"] + "-01")
monthly_volume = monthly_volume.sort_values("month_date")

monthly_volume

## 5. Plot Mortgage Rate And Sales Volume Over Time

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.plot(
    monthly_volume["month_date"],
    monthly_volume["sales_volume"],
    marker="o",
    label="Sales Volume",
)
ax1.set_xlabel("Month")
ax1.set_ylabel("Residential Sold Transactions")

ax2 = ax1.twinx()
ax2.plot(
    monthly_volume["month_date"],
    monthly_volume["rate_30yr_fixed"],
    color="orange",
    marker="o",
    label="30-Year Fixed Mortgage Rate",
)
ax2.set_ylabel("30-Year Fixed Mortgage Rate (%)")

plt.title("Mortgage Rate vs. Residential Sold Volume")
fig.tight_layout()
plt.savefig(FIGURE_DIR / "mortgage_rate_vs_sales_volume_time_series.png")
plt.show()

## 6. Correlation Check

A negative correlation would mean that higher mortgage rates are associated with lower sales volume in this period. This does not prove rates caused the volume change.

In [ ]:
same_month_corr = monthly_volume["rate_30yr_fixed"].corr(monthly_volume["sales_volume"])

monthly_volume["rate_lag_1_month"] = monthly_volume["rate_30yr_fixed"].shift(1)
monthly_volume["rate_lag_2_months"] = monthly_volume["rate_30yr_fixed"].shift(2)

lag_1_corr = monthly_volume["rate_lag_1_month"].corr(monthly_volume["sales_volume"])
lag_2_corr = monthly_volume["rate_lag_2_months"].corr(monthly_volume["sales_volume"])

correlation_summary = pd.DataFrame([
    {"comparison": "same_month_rate_vs_sales_volume", "correlation": same_month_corr},
    {"comparison": "prior_1_month_rate_vs_sales_volume", "correlation": lag_1_corr},
    {"comparison": "prior_2_month_rate_vs_sales_volume", "correlation": lag_2_corr},
])

correlation_summary

## 7. Scatter Plot

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(monthly_volume["rate_30yr_fixed"], monthly_volume["sales_volume"])

for _, row in monthly_volume.iterrows():
    plt.annotate(
        row["year_month"],
        (row["rate_30yr_fixed"], row["sales_volume"]),
        fontsize=8,
        alpha=0.7,
    )

plt.title("Monthly Sales Volume vs. Mortgage Rate")
plt.xlabel("30-Year Fixed Mortgage Rate (%)")
plt.ylabel("Residential Sold Transactions")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "sales_volume_vs_mortgage_rate_scatter.png")
plt.show()

## 8. Compare High-Rate Months Vs. Low-Rate Months

This splits months into above-median and below-median mortgage rate groups, then compares average sales volume.

In [ ]:
median_rate = monthly_volume["rate_30yr_fixed"].median()

monthly_volume["rate_group"] = monthly_volume["rate_30yr_fixed"].apply(
    lambda value: "High-rate months" if value >= median_rate else "Low-rate months"
)

rate_group_summary = (
    monthly_volume.groupby("rate_group")
    .agg(
        month_count=("year_month", "size"),
        average_rate=("rate_30yr_fixed", "mean"),
        average_sales_volume=("sales_volume", "mean"),
        median_sales_volume=("sales_volume", "median"),
    )
    .reset_index()
)

rate_group_summary

## 9. Save Outputs

In [ ]:
monthly_volume.to_csv(REPORT_DIR / "monthly_sales_volume_with_mortgage_rates.csv", index=False)
correlation_summary.to_csv(REPORT_DIR / "mortgage_rate_sales_volume_correlation.csv", index=False)
rate_group_summary.to_csv(REPORT_DIR / "high_vs_low_rate_month_sales_volume.csv", index=False)

print("Saved reports to:", REPORT_DIR)
print("Saved figures to:", FIGURE_DIR)

## 10. Suggested Interpretation Template

Use the tables and charts above to write a cautious summary:

> I aggregated Residential sold records by close month and compared monthly sales volume with the monthly average 30-year fixed mortgage rate. The correlation table shows whether higher rates were associated with higher or lower transaction volume during this period. This is an exploratory relationship, not causal proof, because sales volume is also affected by seasonality, inventory, local demand, pricing, and broader economic conditions.